# Lesson 2: Multi-Turn Conversations with Gemini API

In this notebook, we translate **Lesson 2: Multi-Turn Conversations** from the Claude API Course into Google's **Gemini API (`google-genai`)**.

Like the Anthropic API, Gemini's base `models.generate_content` endpoint is **stateless**. To maintain context across turns, we must pass the entire conversation history with each request.

In [4]:
import os
from dotenv import load_dotenv
from google import genai

# Load API Key
load_dotenv(override=True)

# Initialize Gemini Client
client = genai.Client()
model = "gemini-3.6-flash"


## 1. Building Helper Functions

In Anthropic's API, the assistant role is called `"assistant"` and uses `"content"`.
In Gemini's API:
- The assistant role is named **`"model"`**
- The content structure uses **`"parts"`** with `[{"text": ...}]` or simple dictionaries.

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "parts": [{"text": text}]}
    messages.append(user_message)

def add_assistant_message(messages, text):
    # Note: In Gemini API, assistant role is 'model'
    assistant_message = {"role": "model", "parts": [{"text": text}]}
    messages.append(assistant_message)

def chat(messages):
    response = client.models.generate_content(
        model=model,
        contents=messages,
    )
    return response.text


## 2. Putting It All Together (Manual Context Management)

In [6]:
# Start with an empty message list
messages = []

# Turn 1: Initial user question
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
add_assistant_message(messages, answer)
print("Turn 1 Response:", answer)
print("-" * 50)

# Turn 2: Follow-up question relying on context
add_user_message(messages, "Write another sentence")
final_answer = chat(messages)
add_assistant_message(messages, final_answer)
print("Turn 2 Response:", final_answer)


Turn 1 Response: **Quantum computing** is an advanced field of computer science that harnesses the principles of quantum mechanics—such as superposition and entanglement—to process complex information exponentially faster than classical computers.
--------------------------------------------------
Turn 2 Response: Instead of relying on traditional bits that represent either a 0 or a 1, quantum computers use "qubits" that can exist as both states simultaneously, allowing them to evaluate vast numbers of possibilities all at once.


## 3. Gemini Native Alternative: `client.chats.create`

While the helper functions above show how manual context passing works under the hood, Gemini provides a built-in stateful chat feature so you don't have to manage the array manually:

In [7]:
# Create a native stateful chat session
chat_session = client.chats.create(model=model)

# Turn 1
response1 = chat_session.send_message("Define quantum computing in one sentence")
print("Native Chat Turn 1:", response1.text)
print("-" * 50)

# Turn 2
response2 = chat_session.send_message("Write another sentence")
print("Native Chat Turn 2:", response2.text)


Native Chat Turn 1: Quantum computing is an advanced technology that harnesses the principles of quantum mechanics, such as superposition and entanglement, to solve complex problems far beyond the reach of classical computers.
--------------------------------------------------
Native Chat Turn 2: By evaluating vast numbers of possibilities simultaneously, quantum computers can perform calculations in seconds that would take traditional supercomputers thousands of years.
